In [8]:
import pandas as pd

# Load the dataset
df = pd.read_csv("results.csv")

# Show the first few rows
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False


In [9]:
print(df.columns.tolist())

['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']


In [11]:
import pandas as pd

# Load the dataset
df = pd.read_csv("results.csv")

# Create a result column
def get_result(row):
    if row['home_score'] > row['away_score']:
        return 'Home Win'
    elif row['home_score'] < row['away_score']:
        return 'Away Win'
    else:
        return 'Draw'

df['result'] = df.apply(get_result, axis=1)

# Keep only needed columns
df = df[['home_team', 'away_team', 'home_score', 'away_score', 'result']]
df.dropna(inplace=True)

# Convert categorical values to numerical
df['home_team'] = df['home_team'].astype('category').cat.codes
df['away_team'] = df['away_team'].astype('category').cat.codes
df['result'] = df['result'].astype('category').cat.codes


In [12]:
from sklearn.model_selection import train_test_split

X = df[['home_team', 'away_team', 'home_score', 'away_score']]  # features
y = df['result']  # target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
model.fit(X_train, y_train)

RandomForestClassifier()

In [14]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9998962870773699
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2741
           1       1.00      1.00      1.00      2214
           2       1.00      1.00      1.00      4687

    accuracy                           1.00      9642
   macro avg       1.00      1.00      1.00      9642
weighted avg       1.00      1.00      1.00      9642



In [15]:
import pickle

with open("match_outcome_model.pkl", "wb") as f:
    pickle.dump(model, f)

In [16]:
# Only use team names as input features
X = df[['home_team', 'away_team']]
y = df['result']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [17]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
model.fit(X_train, y_train)

RandomForestClassifier()

In [18]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.5112009956440573
              precision    recall  f1-score   support

           0       0.47      0.44      0.45      2741
           1       0.26      0.17      0.21      2214
           2       0.59      0.71      0.65      4687

    accuracy                           0.51      9642
   macro avg       0.44      0.44      0.44      9642
weighted avg       0.48      0.51      0.49      9642



In [20]:
import pickle

with open("match_outcome_model.pkl", "wb") as f:
    pickle.dump(model, f)

In [26]:
import streamlit as st
import pandas as pd
import pickle

# Load model
with open("match_outcome_model.pkl", "rb") as f:
    model = pickle.load(f)

# Load teams
teams_df = pd.read_csv("results.csv")[['home_team', 'away_team']]
all_teams = pd.unique(teams_df[['home_team', 'away_team']].values.ravel())

st.title("⚽ Football Match Outcome Predictor")

home_team = st.selectbox("Select Home Team", sorted(all_teams))
away_team = st.selectbox("Select Away Team", sorted(all_teams))

if st.button("Predict Outcome"):
    if home_team == away_team:
        st.warning("Please select two different teams.")
    else:
        # ✅ Correct way to encode
        home_code = pd.Categorical([home_team], categories=all_teams).codes[0]
        away_code = pd.Categorical([away_team], categories=all_teams).codes[0]

        prediction = model.predict([[home_code, away_code]])[0]
        outcome_map = {0: "Home Win", 1: "Draw", 2: "Away Win"}
        st.success(f"🏁 Predicted Outcome: **{outcome_map[prediction]}**")


In [28]:
import pickle

with open("match_outcome_model.pkl", "wb") as f:
    pickle.dump(model, f)
#cd C:\Users\sumod\Desktop\ANACONDA
